# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [4]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    #loop up to tries times
    for i in range(tries):
        r = requests.get(BASE, params=params, timeout=5)
        if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])
    #raise on a status that is not in RETRY_STATUSES
    if r.status_code not in RETRY_STATUSES:
        r.raise_for_status()
    #otherwise sleep (increasing each time) and try again
    time.sleep(2**i)
    return get_weather(start, end, lat, lon, tries)
    #raise if every attempt failed
    raise Exception('all attempts failed')

    pass

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [14]:
#call get_weather once for a range that spans 2026-09-05 and 2026-10-03 then keep just the two dates you care about
wx = get_weather('2026-09-05', '2026-10-03')

#keep only 2026-09-05 and 2026-10-03
wx = wx.loc[wx['time'].isin(['2026-09-05', '2026-10-03'])]
print(wx)



used_fallback = False

          time  temperature_2m_max  precipitation_sum
0   2026-09-05                29.4                2.5
28  2026-10-03                22.0               14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [16]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

#if you used the fallback, set used_fallback = True and build `wx` from FALLBACK


if used_fallback:
    wx = FALLBACK
#add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
wx['source'] = 'open-meteo' if not used_fallback else 'fallback sample'
print(wx)

          time  temperature_2m_max  precipitation_sum      source
0   2026-09-05                29.4                2.5  open-meteo
28  2026-10-03                22.0               14.7  open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [17]:
#produce a frame with date, temp_max, precip_mm, source
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
print(wx)

         date  temp_max  precip_mm      source
0  2026-09-05      29.4        2.5  open-meteo
28 2026-10-03      22.0       14.7  open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [18]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [22]:

if wx.loc[wx['precip_mm'].idxmax()]['date'] == '2026-09-05':
    print('September 5th was wetter by', wx.loc[wx['precip_mm'].idxmin()]['precip_mm'] - wx.loc[wx['precip_mm'].idxmax()]['precip_mm'], 'mm')

else:
    print('October 3rd was wetter by', wx.loc[wx['precip_mm'].idxmax()]['precip_mm'] - wx.loc[wx['precip_mm'].idxmin()]['precip_mm'], 'mm')



October 3rd was wetter by 12.2 mm


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [23]:
#print the endpoint, coordinates, date range, when data was pulled, and whether real or fallback data
print('Endpoint:', BASE)
print('Coordinates:', 'lat=38.03, lon=-78.51')
print('Date range:', '2026-09-05 to 2026-10-03')

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: lat=38.03, lon=-78.51
Date range: 2026-09-05 to 2026-10-03


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather comes up empty, and you attempt to merge it onto 200 days of sales charting against revenue, the chart would show null values for the rain values. Nobody would catch it because the merge would work normally, it would just look like there is no correlation between rain and revenue

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [25]:
#find 2026-09-05 and 2026-10-03 data for Orlando with coordinates (28.54, -81.38)

START, END = '2026-09-05', '2026-10-03'

CITIES = {
    'Charlottesville': (38.03, -78.51),
    'Orlando':         (28.54, -81.38),
}

frames = []
for city, (lat, lon) in CITIES.items():
    w = get_weather(START, END, lat, lon)
    frames.append(w.assign(city=city))

weather_long = (
    pd.concat(frames, ignore_index=True)
      .rename(columns={'time': 'date',
                       'temperature_2m_max': 'temp_max',
                       'precipitation_sum': 'rain_mm'})
)
weather_long['date'] = pd.to_datetime(weather_long['date'])

# Guards against the silent-failure problem from earlier
expected = (pd.Timestamp(END) - pd.Timestamp(START)).days + 1
assert len(weather_long) == expected * len(CITIES), 'Unexpected row count'
assert weather_long[['temp_max', 'rain_mm']].notna().all().all(), 'Missing values'
assert not weather_long.duplicated(['city', 'date']).any(), 'Duplicate city/date'

print(weather_long)

         date  temp_max  rain_mm             city
0  2026-09-05      29.4      2.5  Charlottesville
1  2026-09-06      27.2      1.6  Charlottesville
2  2026-09-07      27.7      0.0  Charlottesville
3  2026-09-08      29.4      0.0  Charlottesville
4  2026-09-09      30.9      0.0  Charlottesville
5  2026-09-10      34.2      3.2  Charlottesville
6  2026-09-11      28.0     10.4  Charlottesville
7  2026-09-12      27.5     21.5  Charlottesville
8  2026-09-13      31.3      5.8  Charlottesville
9  2026-09-14      28.7      0.3  Charlottesville
10 2026-09-15      25.8      0.0  Charlottesville
11 2026-09-16      29.7      0.0  Charlottesville
12 2026-09-17      33.7      0.6  Charlottesville
13 2026-09-18      30.0      1.6  Charlottesville
14 2026-09-19      27.1      2.3  Charlottesville
15 2026-09-20      30.5      0.1  Charlottesville
16 2026-09-21      24.7      1.0  Charlottesville
17 2026-09-22      18.9     11.3  Charlottesville
18 2026-09-23      14.9      6.4  Charlottesville
